# Query Engine

After restarting the kernel, run the cells from top to bottom (or use **Run All**) before running a query.

In [20]:
import os
from dotenv import load_dotenv,find_dotenv
env_path = find_dotenv(".env", usecwd=True)
print("Loaded:", load_dotenv(env_path, override=True))
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()

Loaded: True


In [21]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)

In [22]:
from pathlib import Path

pdf_dir = Path("data")
pdf_dir.mkdir(exist_ok=True)
pdf_path = pdf_dir / "transformers.pdf"
if not pdf_path.is_file():
    raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

In [23]:
from llama_index.readers.file import PDFReader
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

documents = PDFReader().load_data(file=pdf_path)
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

In [24]:
retriever = index.as_retriever()

In [25]:
from llama_index.core import get_response_synthesizer
from llama_index.llms.groq import Groq

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Add a Groq API key to the project .env file.")

llm = Groq(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY)

In [26]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from llama_index.readers.file import PDFReader

env_path = find_dotenv(".env", usecwd=True)
if env_path:
    load_dotenv(env_path, override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Add a Groq API key to the project .env file.")

pdf_path = Path("data") / "transformers.pdf"
if not pdf_path.is_file():
    raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

if "documents" not in globals():
    documents = PDFReader().load_data(file=pdf_path)
if "index" not in globals():
    embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
    index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
if "llm" not in globals():
    llm = Groq(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY)

response_synthesizer = get_response_synthesizer(
    llm=llm,
    response_mode="compact",
)
query_engine = index.as_query_engine(
    llm=llm,
    response_synthesizer=response_synthesizer,
    similarity_top_k=10,
)

In [27]:
if "query_engine" not in globals():
    raise RuntimeError("The query engine is not initialized. Run the setup cells from top to bottom, or use Run All.")

response=query_engine.query(
    'Who are the authors of the paper "Attention Is All You Need"? '
    'List every author exactly as written in the paper.'
)
print(response)

Ashish Vaswani  
Noam Shazeer  
Niki Parmar  
Jakob Uszkoreit  
Llion Jones  
Aidan N. Gomez  
Łukasz Kaiser  
Illia Polosukhin
